In [32]:
!pip install requests beautifulsoup4 pandas lxml

In [37]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import json
import re

base_url = "https://www.amazon.eg"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9"
}

urls = [
    "https://www.amazon.eg/-/en/s?k=laptop&rh=p_123%3A308445&dc&crid=1QHV4XI4MD67&qid=1777152752&rnid=91049076031&sprefix=la%2Caps%2C167&ref=sr_nr_p_123_3&ds=v1%3AlJzmSnCnXgsTnnwgMoCHF88HjG2c7m%2BBO2v3GDoHimU",
   # "https://www.amazon.eg/-/en/s?k=laptop&rh=p_123%3A247341%257C391242&dc&crid=1QHV4XI4MD67&qid=1777152586&rnid=91049076031&sprefix=la%2Caps%2C167&ref=sr_nr_p_123_3&ds=v1%3Atn9ynCW4e2XWK0xDYtYzZ5IPEV7PS%2FIQX6XFxfWzVys",
    #"https://www.amazon.eg/s?k=phone+samsung&crid=IXOQR8NQDU26&sprefix=phone+sa%2Caps%2C374&ref=nb_sb_ss_mvt-t11-ranker_1_8",
   #"https://www.amazon.eg/s?k=iphone",
    #"https://www.amazon.eg/-/en/s?k=ipad",
    #"https://www.amazon.eg/s?k=laptop",
    #"https://www.amazon.eg/s?k=phone+huawei&crid=QYAGDNOMQPCX&sprefix=phone+hu%2Caps%2C157&ref=nb_sb_ss_mvt-t11-ranker_1_8",
    #"https://www.amazon.eg/s?k=watches+touch+smart+iphone&crid=1KEDO2JK0J5XF&sprefix=watches+touch+smart+iphone%2Caps%2C142&ref=nb_sb_noss",
   #"https://www.amazon.eg/-/en/s?k=watches+touch+smart+iphone&page=2&xpid=s1uWqkfccQWvg&crid=1KEDO2JK0J5XF&qid=1777097894&sprefix=watches+touch+smart+iphone%2Caps%2C142&ref=sr_pg_2",
   #"https://www.amazon.eg/-/en/s?k=watches+touch+smart+iphone&page=3&crid=1KEDO2JK0J5XF&qid=1777098945&sprefix=watches+touch+smart+iphone%2Caps%2C142&xpid=s1uWqkfccQWvg&ref=sr_pg_3",
#"https://www.amazon.eg/s?k=ipad+samsung&crid=1E1OZ1GIEHOA7&sprefix=ipad+samsung%2Caps%2C172&ref=nb_sb_noss_1",
#"https://www.amazon.eg/s?k=earbuds&crid=GG6ZJR7QLPO&sprefix=ear%2Caps%2C175&ref=nb_sb_ss_mvt-t11-ranker_1_3",

    #"https://www.amazon.eg/s?k=redmi&crid=3R6M8AHYXCPR8&sprefix=re%2Caps%2C172&ref=nb_sb_ss_mvt-t11-ranker_1_2",
    #"https://www.amazon.eg/-/en/s?k=redmi&page=2&xpid=arVaQLVp-Nodq&crid=3R6M8AHYXCPR8&qid=1777099332&sprefix=re%2Caps%2C172&ref=sr_pg_2",
]

data = []
i = 1

for search_url in urls:
    print("Scraping:", search_url)
    response = requests.get(search_url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    products = soup.find_all("div", {"data-component-type": "s-search-result"})

    for product in products:
        # ---------------------------
        # BASIC DATA
        # ---------------------------
        title_tag = product.h2
        title = title_tag.text.strip() if title_tag else "N/A"

        price_tag = product.find("span", class_="a-price-whole")
        price = price_tag.text.strip() if price_tag else "N/A"

        fraction_tag = product.find("span", class_="a-price-fraction")
        price_fraction = fraction_tag.text.strip() if fraction_tag else "00"

        price_clean = price.replace(",", "").replace("..", ".")
        full_price = f"{price_clean}.{price_fraction}" if price != "N/A" else "N/A"

        image_tag = product.find("img")
        image_url = image_tag["src"] if image_tag else "N/A"

        rating_tag = product.find("span", class_="a-icon-alt")
        rating_text = rating_tag.text.strip() if rating_tag else "0"
        product_rating = rating_text.split()[0] if rating_text else "0"

        reviews_tag = product.find("span", class_="a-size-base")
        reviews_raw = reviews_tag.text.strip() if reviews_tag else "0"
        reviews_count = re.findall(r'\d+', reviews_raw)
        reviews_count = reviews_count[0] if reviews_count else "0"

        link_tag = product.find("a", class_="a-link-normal")
        product_url = base_url + link_tag['href'] if link_tag else None

        # ---------------------------
        # PRODUCT PAGE DATA
        # ---------------------------
        description = ""
        seller_name = ""
        seller_rating = ""
        seller_reviews_count = "0"
        original_price = full_price
        discount_percent = 0
        is_verified = False
        reviews_list = []

        if product_url:
            try:
                page = requests.get(product_url, headers=headers, timeout=10)
                soup2 = BeautifulSoup(page.content, "html.parser")

                # DESCRIPTION
                bullets = soup2.select("#feature-bullets li span")
                if bullets:
                    description = " ".join(b.get_text(strip=True) for b in bullets if len(b.get_text(strip=True)) > 10)
                if not description:
                    desc_tag = soup2.find("div", id="productDescription")
                    if desc_tag:
                        description = desc_tag.text.strip()

                # SELLER
                seller_tag = soup2.find("a", id="bylineInfo")
                if seller_tag:
                    seller_name = seller_tag.get_text(strip=True)

                seller_rating_tag = soup2.select_one("#acrPopover")
                if seller_rating_tag and seller_rating_tag.get("title"):
                    seller_rating = seller_rating_tag.get("title").split()[0]

                reviews_tag_seller = soup2.select_one("#acrCustomerReviewText")
                if reviews_tag_seller:
                    reviews_raw = reviews_tag_seller.get_text(strip=True)
                    seller_reviews_count = re.findall(r'[\d,]+', reviews_raw)[0].replace(",", "")

                # ORIGINAL PRICE
                price_tag2 = soup2.select_one("span.a-price.a-text-price span.a-offscreen")
                if price_tag2:
                    original_raw = price_tag2.get_text(strip=True).replace("EGP", "").strip()
                    original_price = re.sub(r'[^\d.]', '', original_raw)

                discount_tag = soup2.find("span", class_="savingsPercentage")
                if discount_tag:
                    discount_raw = discount_tag.get_text(strip=True)
                    discount_percent = re.findall(r'\d+', discount_raw)[0]

                # VERIFIED SELLER
                if seller_name and "Amazon" in seller_name:
                    is_verified = True

                # REVIEWS
                review_blocks = soup2.select(".review-text-content span")
                for r in review_blocks[:5]:
                    txt = r.get_text(strip=True)
                    if len(txt) > 20:
                        reviews_list.append(txt)

            except Exception as e:
                print(f"⚠️ Error on {title[:30]}: {e}")

        # ---------------------------
        # CATEGORY
        # ---------------------------
        title_lower = title.lower()
        url_lower = search_url.lower()
        text = title_lower + " " + url_lower

        if any(k in text for k in ["iphone", "redmi", "samsung", "huawei", "mobile", "phone", "smartphone"]):
            category = "mobile"
        elif any(k in text for k in ["ipad", "tablet"]):
            category = "tablet"
        elif any(k in text for k in ["laptop", "notebook"]):
            category = "laptop"
        elif any(k in text for k in ["watch", "smartwatch"]):
            category = "watch"
        elif any(k in text for k in ["airpods", "headphones", "earbuds"]):
            category = "accessories"
        else:
            category = "other"

        # ---------------------------
        # SAVE DATA (INSIDE THE LOOP)
        # ---------------------------
        data.append({
            "id": i,
            "source": search_url.split("//")[1].split("/")[0].replace("www.", ""),
            "label": "",
            "category": category,
            "title": title,
            "description": description if description else "",
            "price": full_price,
            "original_price": original_price,
            "discount_percent": int(discount_percent) if discount_percent else 0,
            "image_url": image_url,
            "seller_name": seller_name,
            "seller_rating": seller_rating,
            "seller_reviews_count": str(seller_reviews_count),
            "is_verified": is_verified,
            "product_rating": str(product_rating),
            "reviews_count": str(reviews_count),
            "reviews": json.dumps(reviews_list, ensure_ascii=False),
            "location": "Egypt"
        })

        print(f"✅ {i}. {title[:40]} - {full_price} - Rating: {product_rating}")
        i += 1
        time.sleep(1)

    time.sleep(2)



df = pd.DataFrame(data)

# Column order
columns_order = [
    'id', 'source', 'label', 'category', 'title', 'description',
    'price', 'original_price', 'discount_percent', 'image_url',
    'seller_name', 'seller_rating', 'seller_reviews_count', 'is_verified',
    'product_rating', 'reviews_count', 'reviews', 'location'
]

df = df[columns_order]



Scraping: https://www.amazon.eg/-/en/s?k=laptop&rh=p_123%3A308445&dc&crid=1QHV4XI4MD67&qid=1777152752&rnid=91049076031&sprefix=la%2Caps%2C167&ref=sr_nr_p_123_3&ds=v1%3AlJzmSnCnXgsTnnwgMoCHF88HjG2c7m%2BBO2v3GDoHimU
✅ 1. HP OmniBook 3 AI PC15-fn0001ne: Ryzen AI - 36999..00 - Rating: 5.0
✅ 2. HP ProBook 460 G11 Business Laptop 16” F - 89999..00 - Rating: 4.3
✅ 3. HP 15-DW1380NIA Laptop – 15.6″ FHD Intel - 24999..00 - Rating: 0
✅ 4. HP ProBook G11 Laptop Notebook, Intel® C - 97777..00 - Rating: 4.0
✅ 5. HP OmniBook 5 Flip14-fp0025ne i5-1334U,  - 37999..00 - Rating: 0
✅ 6. HP OMEN MAX Gaming Laptop 16-ah0016ne: U - 129999..00 - Rating: 0
✅ 7. Hp Victus 15-fa1041ne Gaming Laptop - 13 - 42999..00 - Rating: 4.4
✅ 8. HP Victus Gaming 15-fb0034ne Laptop - Ry - 44999..00 - Rating: 3.9
✅ 9. HP ProBook 460 G11 Business Laptop 16” F - 73999..00 - Rating: 4.3
✅ 10. HP OmniBook X 14 inch AI Laptop PC, 2.2K - 72999..00 - Rating: 0
✅ 11. HP ProBook 460 Notebook, Intel Core Ultr - 38299..00 - Rating: 0
✅

In [38]:
from openpyxl import Workbook
from openpyxl.drawing.image import Image
import requests
from io import BytesIO

wb = Workbook()
ws = wb.active

# =========================
# HEADERS (كل الأعمدة)
# =========================
ws.append([
    "id","source","label","category","title","description",
    "price","original_price","discount_percent","image_url",
    "seller_name","seller_rating","seller_reviews_count","is_verified",
    "product_rating","reviews_count","reviews","location",
    "image"
])

row = 2

for item in data:

    # =========================
    # write row data
    # =========================
    ws.append([
        item["id"],
        item["source"],
        item["label"],
        item["category"],
        item["title"][:200],
        item["description"][:500],
        item["price"],
        item["original_price"],
        item["discount_percent"],
        item["image_url"],
        item["seller_name"],
        item["seller_rating"],
        item["seller_reviews_count"],
        item["is_verified"],
        item["product_rating"],
        item["reviews_count"],
        str(item["reviews"])[:1000],
        item["location"],
        ""   # image placeholder
    ])

    # =========================
    # IMAGE INSERT
    # =========================
    try:
        img_url = item.get("image_url")

        if img_url and img_url.startswith("http"):
            response = requests.get(img_url, timeout=10, headers=headers)

            img = Image(BytesIO(response.content))
            img.width = 80
            img.height = 80

            ws.add_image(img, f"S{row}")  # column S = image
            ws.row_dimensions[row].height = 85

    except Exception as e:
        print("Image error:", e)

    row += 1

wb.save("48 ✅4.xlsx")
print("Excel saved successfully ✅")

Excel saved successfully ✅


In [39]:
df = pd.DataFrame(data)
(df.tail(30))

,id,source,label,category,title,description,price,original_price,discount_percent,image_url,seller_name,seller_rating,seller_reviews_count,is_verified,product_rating,reviews_count,reviews,location
18,19,amazon.eg,,laptop,Hp Victus 15-fa1038ne Gaming Laptop - 13th Int...,Intel Core i7-13700H (up to 5.0GHz with Intel ...,47199..00,47199..00,0,https://m.media-amazon.com/images/I/71TN45+oJ0...,Visit the HP Store,1.0,1,False,1.0,1,"[""للاسف مكتوب على المنتج license windows proو ...",Egypt
19,20,amazon.eg,,laptop,HP 14 Inch (35.5 cm) Black & Blue Reversible N...,Flexible Style: Its reversible design lets you...,299..00,299..00,0,https://m.media-amazon.com/images/I/81K2fzt4RY...,Visit the HP Store,4.5,934,False,4.5,0,"[""Good quality and truly HP product. Deserve t...",Egypt
20,21,amazon.eg,,laptop,HP Everyday - 14 inch Laptop Briefcase - A08JS...,HP Everyday - 14 inch Laptop Briefcase - A08JS...,1639..00,1639..00,0,https://m.media-amazon.com/images/I/810GIaR5Wp...,Visit the HP Store,4.7,15,False,4.7,0,"[""Excellent choice for slim laptops 14 inch ma...",Egypt
21,22,amazon.eg,,laptop,HP Professional Laptop Backpack 17.3-inch - 50...,HP Professional Laptop Backpack 17.3-inch - 50...,2990..00,2990..00,0,https://m.media-amazon.com/images/I/71zkaTDg0c...,Visit the HP Store,4.7,740,False,4.7,0,"[""الشنطة رائعة جدااشتريتها في عروض الجمعة البي...",Egypt
22,23,amazon.eg,,laptop,HP Star Wars Special Edition Laptop Sleeve - 1...,Dark side inspired: Your notebook will make a ...,299..00,299..00,0,https://m.media-amazon.com/images/I/81eiuYvzMc...,Visit the HP Store,4.7,844,False,4.7,0,"[""مقاس مختلف و الكفر العادي مش ايديشن زي ما حا...",Egypt
23,24,amazon.eg,,laptop,"HP Carrying Case (Sleeve) for 14"" Notebook Bla...","Hp 14"" Neoprene Sleeve (Black/Geometric)",310..00,461.00,33,https://m.media-amazon.com/images/I/71xJ7T4JLk...,Visit the HP Store,4.6,514,False,4.6,0,"[""جميل زي الصوره بالضبط وخامته حلوه وبتمط شويه...",Egypt
24,25,amazon.eg,,laptop,HP 4QF95AA Duotone 15.6in Laptop Briefcase - B...,"Brand: HP, Item Weight: 0.45 Kg, Product Dimen...",790..00,790..00,0,https://m.media-amazon.com/images/I/91URkW21iW...,Visit the HP Store,4.6,248,False,4.6,0,"[""مساحتها حلوة وخامتها جميلة"", ""خامتها حلوه جد...",Egypt
25,26,amazon.eg,,laptop,HP Pavilion Gaming 400 6Eu57Aa Laptop Backpack...,Features a strap to securely hold your headset...,1099..00,1099..00,0,https://m.media-amazon.com/images/I/715jSmx6kK...,Visit the HP Store,4.6,223,False,4.6,1,"[""really it's a good proudact"", ""الخامة والتصم...",Egypt
26,27,amazon.eg,,laptop,HP 6B8U6AA Travel 18L Expandable 15.6 Laptop B...,HP 6B8U6AA Travel 18L Expandable 15.6 Laptop B...,2159..00,2159..00,0,https://m.media-amazon.com/images/I/911bSlBWzd...,Visit the HP Store,4.5,530,False,4.5,0,"[""شكلها شيك وخامتها حلوة.وحجمها متوسط او اكبر ...",Egypt
27,28,amazon.eg,,laptop,HP OMEN Gaming Business Water Resistant Backpa...,OMEN Gaming Backpack 17.3 Inch Black/Red.,1495..00,2124.00,30,https://m.media-amazon.com/images/I/71IDrvtGl7...,Visit the HP Store,4.6,681,False,4.6,0,"[""الشنطة كخامات وتقفيل وكل حاجه فيها كويسه جدا...",Egypt
